In [1]:
# 导入必要的包和模块
from deabook.SBM import SBM  # 导入SBM模型类
from deabook.constant import RTS_CRS, RTS_VRS1  # 导入规模报酬假设常量
import pandas as pd  # 导入pandas数据处理库

# 设置分类变量的类型和顺序
from pandas.api.types import CategoricalDtype
# 定义一个有序分类类型，包含三个类别：MQ, MEFFCH, MTECHCH
cat_size_order = CategoricalDtype(
    ['MQ', 'MEFFCH', 'MTECHCH'], 
    ordered=True
)

# 设置分析类型
kind = "country"  # 可选值为 "province" 或 "country"

# 根据分析类型加载不同的数据集
if kind == "province":
    dmuname = '省份'  # 设置DMU(决策单元)名称为"省份"
    # 读取中国省份数据，筛选2015年以后的数据，重置索引
    data = pd.read_excel(r"../../data\china data.xlsx").query('year>2015').reset_index(drop=True)
    # 提取省份和地区分组信息
    region = data[["省份","group"]]
else:
    dmuname = '国家'  # 设置DMU(决策单元)名称为"国家"
    # 读取OECD国家数据，筛选2015年以后的数据，重置索引
    data = pd.read_excel(r"../../data\oecd data.xlsx").query('year>2015').reset_index(drop=True)

# 提取年份和DMU名称信息
yearid = data[['year', dmuname]]
# 获取所有唯一的DMU名称
yearid[dmuname].unique()

array(['澳大利亚', '奥地利', '比利时', '加拿大', '瑞士', '智利', '捷克', '德国', '丹麦', '西班牙',
       '爱沙尼亚', '芬兰', '法国', '英国', '希腊', '匈牙利', '爱尔兰', '以色列', '意大利', '日本',
       '韩国', '卢森', '墨西哥', '荷兰', '挪威', '新西兰', '波兰', '葡萄牙', '斯洛伐克', '斯洛文尼亚',
       '瑞典', '土耳其', '美国'], dtype=object)

In [2]:
# 描述性统计
data[['K','L','E','Y','CO2']].describe()

,K,L,E,Y,CO2
count,1.320000e+02,132.000000,132.000000,1.320000e+02,132.000000
mean,7.839578e+09,18.345885,7.293261,1.702201e+09,377.733409
std,1.214657e+10,29.280872,17.008826,3.350115e+09,907.427351
min,1.890386e+08,0.414172,0.088174,3.594912e+07,9.092700
25%,1.465476e+09,2.896005,1.043768,2.855821e+08,41.805775
50%,2.747511e+09,4.998875,1.916515,4.895265e+08,84.400650
75%,1.096015e+10,25.462513,6.796940,2.070796e+09,390.508250
max,6.708090e+10,158.299591,101.234876,1.997456e+10,5375.490600


### 非期望产出导向

In [3]:
# 非期望产出导向 SBM（CRS / VRS）
res_bad_crs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[0,0,0], gb=[1], rts=RTS_CRS).optimize(solver="mosek")
res_bad_vrs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[0,0,0], gb=[1], rts=RTS_VRS1).optimize(solver="mosek")

Optimizing locally.
Optimizing locally.


In [4]:
# CRS透视表：按国家x年份
res_bad_crs = res_bad_crs.drop(columns=['optimization_status','direction'])
res_bad_crs2 = pd.concat([res_bad_crs, yearid], axis=1)
res_bad_crs3 = res_bad_crs2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
res_bad_crs3.to_markdown("table8.2_SBM_bad_CRS.md")
res_bad_crs3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.630148,0.646186,0.650721,0.682434
以色列,0.679291,0.698162,0.697001,0.699788
加拿大,0.551829,0.553549,0.554236,0.554816
匈牙利,0.578916,0.578273,0.583295,0.588469
卢森,0.592109,0.591311,0.593698,0.594720
土耳其,0.624967,0.629028,0.628628,0.629602
墨西哥,0.602592,0.612251,0.617545,0.619424
奥地利,0.605576,0.604164,0.613435,0.612510
希腊,0.558410,0.556181,0.559873,0.567528


In [6]:
# CRS透视表：按国家x年份
res_bad_vrs = res_bad_vrs.drop(columns=['optimization_status','direction'])
res_bad_vrs2 = pd.concat([res_bad_vrs, yearid], axis=1)
res_bad_vrs3 = res_bad_vrs2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
res_bad_vrs3.to_markdown("table8.3_SBM_bad_VRS.md")
res_bad_vrs3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.675859,0.696619,0.701107,0.743963
以色列,0.721270,0.741701,0.736179,0.735590
加拿大,0.610617,0.617299,0.620383,0.622286
匈牙利,0.612250,0.609072,0.613653,0.618666
卢森,1.000000,0.986434,0.967702,0.957516
土耳其,0.913301,1.000000,1.000000,0.905338
墨西哥,0.796323,0.837656,0.863924,0.864251
奥地利,0.619323,0.616797,0.626394,0.624786
希腊,0.576630,0.573231,0.577708,0.587338


### 投入导向

In [8]:
# 投入导向 SBM（CRS / VRS）
res_input_crs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[1,1,1], gb=[0], rts=RTS_CRS).optimize(solver="mosek")
res_input_vrs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[1,1,1], gb=[0], rts=RTS_VRS1).optimize(solver="mosek")

Optimizing locally.
Optimizing locally.


In [9]:
# CRS透视表：按国家x年份
res_input_crs = res_input_crs.drop(columns=['optimization_status','direction'])
res_input_crs2 = pd.concat([res_input_crs, yearid], axis=1)
res_input_crs3 = res_input_crs2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
res_input_crs3.to_markdown("table8.4_SBM_input_CRS.md")
res_input_crs3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.589967,0.600560,0.609808,0.623568
以色列,0.645703,0.650815,0.666009,0.681304
加拿大,0.477707,0.482436,0.486656,0.487377
匈牙利,0.441567,0.448944,0.462453,0.474470
卢森,0.605678,0.598844,0.599063,0.599044
土耳其,0.614039,0.622436,0.623598,0.615478
墨西哥,0.511787,0.516624,0.524146,0.515490
奥地利,0.515830,0.519441,0.530020,0.526128
希腊,0.366039,0.365265,0.370826,0.377519


In [10]:
# VRS透视表：按国家x年份
res_input_vrs = res_input_vrs.drop(columns=['optimization_status','direction'])
res_input_vrs2 = pd.concat([res_input_vrs, yearid], axis=1)
res_input_vrs3 = res_input_vrs2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
res_input_vrs3.to_markdown("table8.5_SBM_input_VRS.md")
res_input_vrs3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.621241,0.630172,0.638303,0.650678
以色列,0.698611,0.699761,0.712155,0.724598
加拿大,0.630097,0.637743,0.645484,0.649113
匈牙利,0.471816,0.476982,0.487954,0.497868
卢森,1.000000,0.979910,0.965664,0.957258
土耳其,0.970238,1.000000,1.000000,0.973265
墨西哥,0.771076,0.780829,0.794835,0.777968
奥地利,0.516438,0.519495,0.530023,0.526341
希腊,0.388755,0.386617,0.391548,0.397493


### 结果汇总

均值为按国家跨年平均效率。

In [11]:
# 各模型效率汇总
summary = pd.DataFrame({
    '模型': ['非期望产出导向_CRS', '非期望产出导向_VRS', '投入导向_CRS', '投入导向_VRS',],
    '均值': [res_bad_crs['rho'].mean(), res_bad_vrs['rho'].mean(),
            res_input_crs['rho'].mean(), res_input_vrs['rho'].mean(),
            ],
    '标准差': [res_bad_crs['rho'].std(), res_bad_vrs['rho'].std(),
             res_input_crs['rho'].std(), res_input_vrs['rho'].std(),
             ],
    '最大值': [res_bad_crs['rho'].max(), res_bad_vrs['rho'].max(),
             res_input_crs['rho'].max(), res_input_vrs['rho'].max(),
             ],
    '最小值': [res_bad_crs['rho'].min(), res_bad_vrs['rho'].min(),
             res_input_crs['rho'].min(), res_input_vrs['rho'].min(),
            ],
})
summary.round(4)

,模型,均值,标准差,最大值,最小值
0,非期望产出导向_CRS,0.6398,0.1086,1.0,0.5293
1,非期望产出导向_VRS,0.7557,0.1485,1.0,0.5567
2,投入导向_CRS,0.5644,0.1517,1.0,0.3653
3,投入导向_VRS,0.7168,0.1986,1.0,0.3866


### 结果说明

1. **VRS 与 CRS 的比较**：VRS 下两种方向的平均效率均明显高于 CRS，这是因为 VRS 放宽了规模报酬约束。
2. **非期望产出导向 vs 投入导向**：非期望产出导向的平均效率高于投入导向，说明从 CO2 削减潜力角度衡量的效率总体上高于仅从投入节约角度衡量的效率。
3. **有效单元**：VRS 下爱沙尼亚在两种方向上均达到完全有效（ρ=1）。

### 能源投入+非期望产出方向 SBM

同时减少能源投入（E）和碳排放（CO2），资本和劳动力保持不变。

In [37]:
# 能源投入+非期望产出方向 SBM（CRS）
# gx=[0,0,1] 只减少能源投入E，gb=[1] 减少CO2
res_energy_bad_crs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[0,0,1], gb=[1], rts=RTS_CRS).optimize(solver="mosek")
res_energy_bad_crs

Optimizing locally.


,optimization_status,direction,objective_value,rho,t,slack_x_K,slack_x_L,slack_x_E,slack_y_Y,slack_b_CO2
0,ok,hyper_orientedxb,0.185983,0.185983,0.575808,0.0,0.0,4.068332,0.0,306.422205
1,ok,hyper_orientedxb,0.186491,0.186491,0.578656,0.0,0.0,4.201976,0.0,301.998053
2,ok,hyper_orientedxb,0.192288,0.192288,0.580349,0.0,0.0,4.119806,0.0,300.811498
3,ok,hyper_orientedxb,0.192951,0.192951,0.580416,0.0,0.0,4.087682,0.0,299.655220
4,ok,hyper_orientedxb,0.285415,0.285415,0.675086,0.0,0.0,0.843584,0.0,32.350321
...,...,...,...,...,...,...,...,...,...,...
127,ok,hyper_orientedxb,0.368697,0.368697,0.672152,0.0,0.0,2.983749,0.0,194.784081
128,ok,hyper_orientedxb,0.247615,0.247615,0.650457,0.0,0.0,60.311775,0.0,2820.185111
129,ok,hyper_orientedxb,0.264989,0.264989,0.664070,0.0,0.0,58.689934,0.0,2634.425012
130,ok,hyper_orientedxb,0.279339,0.279339,0.675831,0.0,0.0,59.391782,0.0,2578.406848


In [38]:
# 能源投入+非期望产出方向 SBM（VRS）
res_energy_bad_vrs = SBM(data, sent="K L E=Y:CO2", gy=[0], gx=[0,0,1], gb=[1], rts=RTS_VRS1).optimize(solver="mosek")
res_energy_bad_vrs

Optimizing locally.


,optimization_status,direction,objective_value,rho,t,slack_x_K,slack_x_L,slack_x_E,slack_y_Y,slack_b_CO2
0,ok,hyper_orientedxb,0.347103,0.347103,0.633723,0.0,0.0,2.717886,0.0,240.406952
1,ok,hyper_orientedxb,0.364249,0.364249,0.644570,0.0,0.0,2.696442,0.0,228.702582
2,ok,hyper_orientedxb,0.381620,0.381620,0.650001,0.0,0.0,2.543923,0.0,223.999465
3,ok,hyper_orientedxb,0.355147,0.355147,0.639368,0.0,0.0,2.722007,0.0,233.805525
4,ok,hyper_orientedxb,0.288627,0.288627,0.679449,0.0,0.0,0.840644,0.0,31.710942
...,...,...,...,...,...,...,...,...,...,...
127,ok,hyper_orientedxb,0.902319,0.902319,0.905338,0.0,0.0,0.022041,0.0,41.755220
128,ok,hyper_orientedxb,0.880511,0.880511,0.926980,0.0,0.0,4.881808,0.0,413.394273
129,ok,hyper_orientedxb,0.928281,0.928281,0.955044,0.0,0.0,2.736674,0.0,245.141450
130,ok,hyper_orientedxb,0.926910,0.926910,0.956470,0.0,0.0,3.128707,0.0,244.646751


In [ ]:
# CRS透视表：按国家x年份
EB_CRS = res_energy_bad_crs.drop(columns=['optimization_status','direction'])
EB_CRS2 = pd.concat([EB_CRS, yearid], axis=1)
EB_CRS3 = EB_CRS2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
EB_CRS3.to_markdown("table8.7_SBM_energy_bad_CRS.md")
EB_CRS3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.395162,0.423292,0.442677,0.500610
以色列,0.336621,0.354037,0.383294,0.409901
加拿大,0.096816,0.097580,0.098731,0.100676
匈牙利,0.213049,0.212123,0.222821,0.236984
卢森,0.271391,0.266736,0.264256,0.268048
土耳其,0.347312,0.350700,0.370129,0.368697
墨西哥,0.280552,0.292510,0.307837,0.307919
奥地利,0.285415,0.284877,0.306187,0.297546
希腊,0.208446,0.201079,0.206996,0.216358


In [ ]:
# VRS透视表：按国家x年份
EB_VRS = res_energy_bad_vrs.drop(columns=['optimization_status','direction'])
EB_VRS2 = pd.concat([EB_VRS, yearid], axis=1)
EB_VRS3 = EB_VRS2.pivot_table(values='rho', index=dmuname, columns='year').round(6)
EB_VRS3.to_markdown("table8.8_SBM_energy_bad_VRS.md")
EB_VRS3

year,2016,2017,2018,2019
国家,,,,
丹麦,0.446079,0.478877,0.498694,0.570022
以色列,0.412480,0.430099,0.457423,0.481377
加拿大,0.165241,0.171014,0.175181,0.179076
匈牙利,0.242613,0.238773,0.248514,0.262366
卢森,1.000000,0.960605,0.915556,0.905969
土耳其,0.913301,1.000000,1.000000,0.902319
墨西哥,0.649734,0.708946,0.775767,0.743397
奥地利,0.288627,0.286898,0.307114,0.297726
希腊,0.225658,0.216856,0.223065,0.233332


## 绘图

In [53]:
# 数据整理：提取SBM效率值，合并年份和DMU信息
SBM_bad_VRS = res_energy_bad_vrs.drop(columns=['optimization_status']).rename(columns={'rho': 'SBM_env'})
SBM_bad_VRS2 = pd.concat([SBM_bad_VRS, yearid], axis=1)

# 构建透视表：行为国家/省份，列为年份
SBM_bad_VRS3 = SBM_bad_VRS2.pivot_table(
    values=['SBM_env'],
    index=dmuname,
    columns='year',
    aggfunc='mean',
    fill_value=None,
    dropna=True
)
SBM_bad_VRS3

SBM_env                              
year       2016      2017      2018      2019
国家                                           
丹麦     0.446079  0.478877  0.498694  0.570022
以色列    0.412480  0.430099  0.457423  0.481377
加拿大    0.165241  0.171014  0.175181  0.179076
匈牙利    0.242613  0.238773  0.248514  0.262366
卢森     1.000000  0.960605  0.915556  0.905969
土耳其    0.913301  1.000000  1.000000  0.902319
墨西哥    0.649734  0.708946  0.775767  0.743397
奥地利    0.288627  0.286898  0.307114  0.297726
希腊     0.225658  0.216856  0.223065  0.233332
德国     0.576155  0.609457  0.666854  0.728847
意大利    0.574134  0.589102  0.598299  0.620046
挪威     0.211009  0.221326  0.225190  0.250395
捷克     0.172478  0.170289  0.177880  0.187348
斯洛伐克   0.217358  0.214161  0.223914  0.241710
斯洛文尼亚  0.422160  0.426005  0.435510  0.482257
新西兰    0.829082  0.902421  1.000000  0.810363
日本     0.517926  0.558908  0.576821  0.606384
智利     0.232071  0.230527  0.235340  0.235710
比利时    0.166847  0.170426  0.173914  0.178166
法国     0.524911  0.544000  0.584350  0.621716
波兰     0.686923  0.717809  0.831701  1.000000
澳大利亚   0.347103  0.364249  0.381620  0.355147
爱尔兰    0.734995  0.860202  1.000000  1.000000
爱沙尼亚   1.000000  1.000000  1.000000  1.000000
瑞典     0.302861  0.308972  0.335618  0.343049
瑞士     0.876444  0.927031  0.993996  1.000000
美国     0.880511  0.928281  0.926910  1.000000
芬兰     0.191741  0.204877  0.203445  0.217068
英国     0.838933  0.901077  0.927828  1.000000
荷兰     0.264571  0.275226  0.297244  0.310856
葡萄牙    0.274304  0.278805  0.295960  0.319211
西班牙    0.467541  0.477627  0.493067  0.536258
韩国     0.206685  0.212571  0.217896  0.225549

In [54]:
# 计算分年份和分国家的SBM环境效率均值
sbm_mean_year = SBM_bad_VRS3['SBM_env'].mean()  # 各年份的环境效率均值
sbm_mean_cou = SBM_bad_VRS3['SBM_env'].mean(axis=1).sort_values()  # 各国家/省份的环境效率均值（排序）

In [55]:
sbm_mean_year

year
2016    0.480621
2017    0.502588
2018    0.527414
2019    0.540778
dtype: float64

In [56]:
xtick2 = tuple(sbm_mean_year.index)  # 提取年份作为x轴刻度标签
xtick2

(2016, 2017, 2018, 2019)

In [57]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib notebook
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['STSong']  # 设置中文字体为华文宋体
rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

import numpy as np

In [58]:
fig, ax = plt.subplots(figsize=(8,4))  # 创建图形网格

ax.spines['top'].set_visible(False)  # 隐藏顶部边框
ax.spines['right'].set_visible(False)  # 隐藏右侧边框

################################ 绘图 #################################

n_groups = len(xtick2)  # 年份数量
index = np.arange(n_groups)  # x轴位置

# 绘制SBM环境效率趋势线
rects1 = ax.plot(index, sbm_mean_year,
                color='darkblue', alpha=0.8, linestyle="dashed", marker="s", label='SBM环境效率趋势')

###########################################

ax.set_ylabel('SBM环境效率')
ax.set_xticks(index)
ax.set_xticklabels(xtick2)
plt.legend()  # 显示图例

plt.xticks(rotation=80)  # x轴刻度添加80度旋转
if kind == "province":
    plt.savefig("chap8分年份SBM环境效率-省.png", dpi=900, bbox_inches='tight')
else:
    plt.savefig("chap8分年份SBM环境效率-国.png", dpi=900, bbox_inches='tight')
plt.show()

<IPython.core.display.Javascript object>

In [59]:
sbm_mean_cou

国家
比利时      0.172338
加拿大      0.172628
捷克       0.176999
芬兰       0.204283
韩国       0.215675
斯洛伐克     0.224286
希腊       0.224728
挪威       0.226980
智利       0.233412
匈牙利      0.248066
荷兰       0.286974
葡萄牙      0.292070
奥地利      0.295091
瑞典       0.322625
澳大利亚     0.362030
斯洛文尼亚    0.441483
以色列      0.445345
西班牙      0.493623
丹麦       0.498418
日本       0.565010
法国       0.568744
意大利      0.595395
德国       0.645328
墨西哥      0.719461
波兰       0.809108
新西兰      0.885466
爱尔兰      0.898799
英国       0.916959
美国       0.933925
卢森       0.945532
瑞士       0.949368
土耳其      0.953905
爱沙尼亚     1.000000
dtype: float64

In [60]:
xtick2 = tuple(sbm_mean_cou.index)  # 获取排序后的国家/省份名称
xtick2

('比利时',
 '加拿大',
 '捷克',
 '芬兰',
 '韩国',
 '斯洛伐克',
 '希腊',
 '挪威',
 '智利',
 '匈牙利',
 '荷兰',
 '葡萄牙',
 '奥地利',
 '瑞典',
 '澳大利亚',
 '斯洛文尼亚',
 '以色列',
 '西班牙',
 '丹麦',
 '日本',
 '法国',
 '意大利',
 '德国',
 '墨西哥',
 '波兰',
 '新西兰',
 '爱尔兰',
 '英国',
 '美国',
 '卢森',
 '瑞士',
 '土耳其',
 '爱沙尼亚')

In [61]:
fig, ax = plt.subplots(figsize=(12,6))  # 创建图形网格

ax.spines['top'].set_visible(False)  # 隐藏顶部边框
ax.spines['right'].set_visible(False)  # 隐藏右侧边框

################################ 绘图 #################################
if kind == "province":
    n_groups = 30
else:
    n_groups = 33

index = np.arange(n_groups)  # x轴位置
bar_width = 0.5  # 柱子宽度
error_config = {'ecolor': 'black', 'capsize': 2}  # 设置误差线

# 绘制各国SBM环境效率柱状图
rects2 = ax.bar(index, sbm_mean_cou,
                bar_width/2,
                color='olivedrab', alpha=0.8,
                error_kw=error_config,
                edgecolor='white', label='SBM环境效率')

###########################################

ax.set_ylabel('SBM环境效率')
ax.set_xticks(index)
ax.set_xticklabels(xtick2)
plt.legend(loc='upper left')  # 左上角图例

plt.xticks(rotation=80)  # x轴刻度添加80度旋转
if kind == "province":
    plt.savefig("chap8分省份SBM环境效率.png", dpi=900, bbox_inches='tight')
else:
    plt.savefig("chap8分国家SBM环境效率.png", dpi=900, bbox_inches='tight')

plt.show()

<IPython.core.display.Javascript object>